# QLoRA Fine-Tuning: Adapting Qwen3 to the Finance Domain

**Goal:** fine-tune a small open LLM on finance instruction/QA data using **QLoRA** (4-bit quantized base model + LoRA adapters), on a **free Colab T4 GPU (16GB)**.

**What you'll learn:**
1. Why QLoRA makes fine-tuning large models possible on consumer/free GPUs
2. How to load a model in 4-bit and attach LoRA adapters with `peft`
3. How to format a dataset for supervised fine-tuning (SFT)
4. How to train with `trl`'s `SFTTrainer` under tight VRAM
5. How to compare the model's behavior before vs. after fine-tuning

**Before you start:** In Colab, go to `Runtime > Change runtime type` and select **T4 GPU**. If you've been debugging in an existing session, do a clean `Runtime > Restart session` before this run — earlier attempts may have left monkey-patched objects in memory.

> **Model choice note:** This notebook defaults to `Qwen/Qwen3-4B-Instruct-2507` (non-thinking variant) because it reliably fits and trains on a *free* T4 shared by many students at once. If you have **Colab Pro (A100/L4)**, switch `MODEL_NAME` in the config cell to `Qwen/Qwen3-8B` for a stronger result — everything else in the notebook stays the same.

## 1. Install dependencies

In [ ]:
!pip install -q -U "transformers>=4.56.2,<5.0.0" "accelerate>=0.34.0" "peft>=0.13.0" "bitsandbytes>=0.44.0" "trl>=0.20.0" "datasets>=2.20.0" huggingface_hub

In [ ]:
import transformers, trl, peft, bitsandbytes
print("transformers:", transformers.__version__)
print("trl:         ", trl.__version__)
print("peft:        ", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

# If these look older than expected right after the install cell above, Colab likely
# still has a pre-imported version active. Runtime > Restart session, then re-run
# from the top (the pip install will be a no-op / fast since packages are cached).

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found — set Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Config

All the knobs you might want to tweak live here.

In [ ]:
# Base model. Swap to "Qwen/Qwen3-8B" if you have Colab Pro (A100/L4) instead of free T4.
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Public finance instruction dataset (Alpaca-style: instruction / input / output)
DATASET_NAME = "gbharti/finance-alpaca"

# Subsample size — keeps a classroom session inside ~20-30 min of training on a T4.
# Increase this if you have more time / a stronger GPU.
NUM_TRAIN_EXAMPLES = 2000

MAX_SEQ_LENGTH = 512      # finance QA pairs are short; keep this small to save VRAM
OUTPUT_DIR = "qwen3-finance-qlora"
SEED = 42

## 3. Load the base model in 4-bit (QLoRA quantization)

- `nf4`: the 4-bit data type QLoRA introduced — better suited to normally-distributed weights than plain int4.
- `bnb_4bit_use_double_quant`: quantizes the quantization constants too, saving a bit more memory for free.
- We use **bfloat16** as the compute dtype throughout (quantization compute dtype, model load dtype, and LoRA adapter dtype), and train with `bf16=True` instead of `fp16=True`. This is a deliberate choice for this environment: bf16 training skips `GradScaler` entirely (no loss scaling needed for bf16's dynamic range), which avoids a class of dtype-mismatch crashes we hit when mixing fp16 training with a model stack (`transformers`/`trl`/`peft`) that kept defaulting parts of the model back to bfloat16. T4 has no bf16 tensor-core acceleration, so this runs somewhat slower than true fp16 would, but it works correctly — unlike the missing fp16-AMP-scaler kernel, general bf16 arithmetic is implemented on Turing GPUs.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False  # required for gradient checkpointing during training

## 4. Baseline: see how the model answers finance questions *before* fine-tuning

We'll reuse this exact prompt after training so you can compare outputs directly.

In [ ]:
EVAL_PROMPT = "What is the difference between a traditional IRA and a Roth IRA?"


def generate(prompt, model, tokenizer, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,  # keep outputs short/direct for this demo
        return_tensors="pt",
        return_dict=True,  # force a BatchEncoding so attention_mask is included
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)


baseline_answer = generate(EVAL_PROMPT, model, tokenizer)
print("=== BEFORE fine-tuning ===")
print(baseline_answer)

## 5. Load and format the finance dataset

`gbharti/finance-alpaca` is Alpaca-style: each row has `instruction`, `input` (optional context), and `output`. We convert each row into a chat-formatted training string using Qwen's own chat template, so the model learns in the exact format it will be prompted with at inference time.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_NAME, split="train")
raw_dataset = raw_dataset.shuffle(seed=SEED).select(range(NUM_TRAIN_EXAMPLES))
print(raw_dataset)
print(raw_dataset[0])

In [ ]:
def format_example(example):
    user_content = example["instruction"]
    if example.get("input"):
        user_content += f"\n\n{example['input']}"

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}


train_dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
print(train_dataset[0]["text"])

## 6. Attach LoRA adapters

- `r=16`, `lora_alpha=32`: a common, well-tested ratio (alpha = 2x rank). Higher `r` = more trainable capacity but more VRAM/compute.
- `target_modules`: all attention projections (`q/k/v/o_proj`) and MLP projections (`gate/up/down_proj`) — adapting both attention and feed-forward layers gives noticeably better results than attention-only for domain adaptation.
- `prepare_model_for_kbit_training` handles the numerically-sensitive bits (e.g. casting norm layers, enabling gradient checkpointing) so a 4-bit model can be trained safely.
- We then force every trainable (LoRA) parameter to `bfloat16` explicitly, to guarantee it matches the rest of the model regardless of what dtype `peft` decided to create the adapter in.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.bfloat16)

## 7. Training configuration (tuned for a 16GB T4)

- `per_device_train_batch_size=1` + `gradient_accumulation_steps=16`: effective batch size of 16 while keeping per-step VRAM low.
- `bf16=True` / `fp16=False`: bf16 training needs no `GradScaler`, which sidesteps the dtype-mismatch crash we hit forcing fp16 (see the note in section 3).
- `optim="paged_adamw_8bit"`: an 8-bit optimizer with CPU paging — the standard QLoRA optimizer choice, dramatically cuts optimizer-state VRAM.
- `gradient_checkpointing=True`: trades compute for memory by recomputing activations during the backward pass instead of storing them.
- `loss_type="nll"`: `trl`'s default `"chunked_nll"` loss patches the model to skip the `lm_head` matmul for memory efficiency, but that patched path doesn't support PEFT-wrapped 4-bit models — it raises `AttributeError: 'CausalLMOutputWithPast' object has no attribute 'last_hidden_state'`. Forcing the standard `"nll"` loss avoids that broken path entirely.
- The kwarg-filtering below keeps this notebook working across different `trl`/`transformers` releases, which vary in exactly which `SFTConfig`/`TrainingArguments` fields they expose (e.g. `max_length` vs `max_seq_length`, `processing_class` vs `tokenizer`, presence of `warmup_ratio`).

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    seed=SEED,
    loss_type="nll",
)

supported = set(inspect.signature(SFTConfig.__init__).parameters)
if "max_length" not in supported and "max_seq_length" in supported:
    sft_kwargs["max_seq_length"] = sft_kwargs.pop("max_length")
dropped = [k for k in sft_kwargs if k not in supported]
sft_kwargs = {k: v for k, v in sft_kwargs.items() if k in supported}
if dropped:
    print(f"Note: installed trl's SFTConfig doesn't support {dropped}, skipping them.")

sft_config = SFTConfig(**sft_kwargs)
sft_config.loss_type = "nll"  # force it regardless of whether it survived filtering above
print("loss_type in effect:", sft_config.loss_type)

trainer_kwargs = dict(model=model, args=sft_config, train_dataset=train_dataset)
trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters)
trainer_kwargs["processing_class" if "processing_class" in trainer_params else "tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

In [ ]:
trainer.train()

## 8. Save the LoRA adapter

Note we're only saving the small adapter weights (a few MB to ~100MB), not the full base model — that's the point of LoRA.

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to ./{OUTPUT_DIR}")

## 9. Compare: before vs. after fine-tuning

Same prompt, same model instance (it now has the trained adapter active).

In [ ]:
model.eval()
model.config.use_cache = True

finetuned_answer = generate(EVAL_PROMPT, model, tokenizer)

print("=== BEFORE fine-tuning ===")
print(baseline_answer)
print("\n=== AFTER fine-tuning ===")
print(finetuned_answer)

## 10. (Optional) Push the adapter to the Hugging Face Hub

Requires a free HF account and an access token (Settings > Access Tokens). Skip this cell if you just want to keep the adapter local/in Colab.

In [ ]:
PUSH_TO_HUB = False  # flip to True and fill in HUB_REPO_ID to enable
HUB_REPO_ID = "your-username/qwen3-finance-qlora"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # paste your HF token when prompted
    model.push_to_hub(HUB_REPO_ID)
    tokenizer.push_to_hub(HUB_REPO_ID)

## Exercises for students

1. **Change the prompt** in `EVAL_PROMPT` to a finance question of your own and re-run the comparison cell.
2. **Increase `NUM_TRAIN_EXAMPLES`** (e.g. to 5000) and observe how training time and answer quality change.
3. **Try a different LoRA rank** (`r=8` vs `r=32`) — how does `model.print_trainable_parameters()` change, and does answer quality change?
4. **Swap the dataset** to another public finance dataset on the Hub (e.g. `virattt/financial-qa-10K`) and adjust `format_example` to match its columns.
5. **Merge the adapter into the base model** for deployment: `merged = model.merge_and_unload()` — try it and inspect `merged` (note: this needs the model loaded outside 4-bit quantization to save a standalone full-precision merged model).